In [8]:
import rasterio
from rasterio.windows import Window
import numpy as np
import cupy as cp
from tqdm import tqdm
import gc # Garbage Collector

TILE_SIZE = 2048 # Balanced size: big enough for GPU, small enough for RAM

def main():
    red_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B04_10m_converted.tif"
    nir_path = r"C:\Users\hsgpi\OneDrive\Desktop\S2C_MSIL2A_20260213T045921_N0512_R119_T44QQE_20260213T133817.SAFE\GRANULE\L2A_T44QQE_A007525_20260213T051019\IMG_DATA\R10m\B08_10m_converted.tif"
    
    with rasterio.open(red_path) as src:
        h, w = src.height, src.width
        profile = src.profile.copy()
        profile.update(dtype="float32", count=1, tiled=True, compress="lzw", blockxsize=256, blockysize=256)
        
        windows = [Window(c, r, min(TILE_SIZE, w - c), min(TILE_SIZE, h - r)) 
                   for r in range(0, h, TILE_SIZE) for c in range(0, w, TILE_SIZE)]

    with rasterio.open(red_path) as r_s, rasterio.open(nir_path) as n_s, rasterio.open("NDVI_Safe.tif", "w", **profile) as dst:
        for win in tqdm(windows, desc="RAM-Safe GPU Processing"):
            # 1. Read directly into float32 to save memory
            red = r_s.read(1, window=win).astype("float32")
            nir = n_s.read(1, window=win).astype("float32")
            
            # 2. Transfer to GPU
            r_gpu = cp.asarray(red)
            n_gpu = cp.asarray(nir)
            
            # 3. Calculate NDVI
            ndvi_gpu = (n_gpu - r_gpu) / (n_gpu + r_gpu + 1e-6)
            
            # 4. Move back to CPU and write
            dst.write(cp.asnumpy(ndvi_gpu), 1, window=win)
            
            # 5. EXPLICIT CLEANUP: This is the important part for your 16GB RAM
            del red, nir, r_gpu, n_gpu, ndvi_gpu
            gc.collect() # Force RAM to empty
            cp.get_default_memory_pool().free_all_blocks() # Force GPU VRAM to empty

if __name__ == "__main__":
    main()

RAM-Safe GPU Processing: 100%|██████████| 36/36 [00:06<00:00,  5.85it/s]
